# 07d — Consultas espaciales MT: clustering y buffers

Cuarto bloque corto: agrupamiento espacial y áreas de operación.

> Esta notebook es una parte del bloque 07. Está separada para que sea más liviana de editar, ejecutar y revisar en clase.

## 1. Objetivo y prerequisitos

- Tener levantado PostGIS con `docker compose up -d postgis`.
- Haber ejecutado la notebook 06 para publicar `gis.mt_cables`, `gis.mt_postes` y `gis.mt_seccionadores`.
- Ejecutar las celdas en orden. Cada consulta deja el SQL visible antes del resultado.

## 2. Preparación de conexión

In [ ]:
# pathlib permite ubicar la raíz del proyecto aunque Jupyter se abra desde otra carpeta.
from pathlib import Path

# os permite leer variables de entorno para parametrizar la conexión local a PostGIS.
import os

# pandas se usa solo para mostrar resultados tabulares de forma cómoda en Jupyter.
import pandas as pd

# psycopg es el driver moderno de PostgreSQL usado por el proyecto.
try:
    import psycopg
    PSYCOPG_DISPONIBLE = True
except ModuleNotFoundError:
    psycopg = None
    PSYCOPG_DISPONIBLE = False
    print('No está instalado el paquete psycopg en este kernel.')
    print('Ejecutar desde la raíz del proyecto: python3 -m pip install -r requirements.txt')
    print('Después de instalar, reiniciar el kernel de Jupyter y volver a ejecutar la notebook.')

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Cargamos .env local sin pisar variables ya definidas por el entorno.
def cargar_env_local() -> None:
    env_path = RAIZ / '.env'
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

cargar_env_local()

DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'port': int(os.getenv('POSTGRES_PORT', '5432')),
    'dbname': os.getenv('POSTGRES_DB', 'ceml_gis'),
    'user': os.getenv('POSTGRES_USER', 'ceml'),
    'password': os.getenv('POSTGRES_PASSWORD', 'ceml_admin_2026'),
}

SRID_PUBLICACION = 32721

# Esta función corta con una instrucción clara si falta el driver de PostgreSQL.
def exigir_psycopg() -> None:
    if not PSYCOPG_DISPONIBLE:
        raise RuntimeError(
            'Falta instalar psycopg en este kernel. '
            'Ejecutar: python -m pip install -r requirements.txt, '
            'reiniciar el kernel y volver a correr la notebook.'
        )

# Ejecuta una consulta SELECT y devuelve un DataFrame.
# El autocommit evita dejar transacciones abiertas durante ejercicios de solo lectura.
def ejecutar_sql(sql: str) -> pd.DataFrame:
    exigir_psycopg()
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            columnas = [col.name for col in cur.description]
            filas = cur.fetchall()
    return pd.DataFrame(filas, columns=columnas)

print('Raíz del proyecto:', RAIZ)
print('Base local:', DB_CONFIG['dbname'])
print('Host local:', DB_CONFIG['host'])
print('Puerto local:', DB_CONFIG['port'])
print('Usuario local:', DB_CONFIG['user'])
print('SRID de publicación:', SRID_PUBLICACION)

## 3. Verificación de capas GIS

In [ ]:
# Verificamos que estén publicadas las capas finales que usan estas consultas.
SQL_VERIFICAR_CAPAS = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'gis'
  AND table_name IN ('mt_cables', 'mt_postes', 'mt_seccionadores')
ORDER BY table_name;
"""

ejecutar_sql(SQL_VERIFICAR_CAPAS)

## 4. Consulta 9: Agrupamiento espacial de postes

**Consulta espacial:** Clustering ST_ClusterDBSCAN  
**Función:** Detectar concentraciones de postes.  
**Consigna:** Agrupar postes que estén próximos entre sí, usando una distancia máxima de 80 metros y un mínimo de 5 postes por grupo.

**Idea clave:** ST_ClusterDBSCAN identifica grupos espaciales de postes próximos. eps := 80 define la distancia máxima entre vecinos y minpoints := 5 exige al menos cinco postes para formar grupo.

In [ ]:
SQL_CONSULTA_09 = """
SELECT
    cid AS grupo,
    count(*) AS cantidad_postes
FROM (
    SELECT
        p.id,
        ST_ClusterDBSCAN(
            p.geom,
            eps := 80,
            minpoints := 5
        ) OVER () AS cid
    FROM gis.mt_postes p
) q
WHERE cid IS NOT NULL
GROUP BY cid
ORDER BY cantidad_postes DESC;
"""

print(SQL_CONSULTA_09)

In [ ]:
ejecutar_sql(SQL_CONSULTA_09)

## 5. Consulta 10: Área de operación alrededor de seccionadores

**Consulta espacial:** Generación de áreas ST_Buffer  
**Función:** Crear zonas de influencia.  
**Consigna:** Generar un área de operación de 50 metros alrededor de cada seccionador.

**Idea clave:** ST_Buffer crea una geometría poligonal alrededor de cada seccionador. Como la capa está en 32721, el valor 50 representa 50 metros. En la notebook se muestra una vista previa WKT para que el resultado sea legible en tabla.

In [ ]:
SQL_CONSULTA_10 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    round(ST_Area(ST_Buffer(s.geom, 50))::numeric, 2) AS area_m2,
    left(ST_AsText(ST_Buffer(s.geom, 50)), 160) || '...' AS area_operacion_wkt
FROM gis.mt_seccionadores s;
"""

print(SQL_CONSULTA_10)

In [ ]:
ejecutar_sql(SQL_CONSULTA_10)

## Cierre

Si este bloque ejecutó correctamente, continuá con la siguiente parte 07. La separación evita notebooks largas y facilita retomar desde el punto exacto donde quedó la clase.